# Full FP-NEB and Static FP Evaluations on DFT-NEB Images

## 1. Purpose and Protocol Definitions

Generates and runs two protocols for every (foundation potential, pathway)
combination in a canonical DFT-reference dataset:

| Protocol | Description | Input structures |
|---|---|---|
| `full_fp_neb` | Foundation-potential relaxation of both endpoints, followed by climbing-image FP-NEB | Unrelaxed source endpoint structures |
| `fp_static_on_dft_neb` | Static (single-point) foundation-potential evaluation, no relaxation | Finalized DFT-NEB image structures |

These two input sources are never interchanged. Neither protocol uses
endpoint-correspondence metadata to reorder or relabel structures.

Endpoint-role vocabulary (matches the canonical reference and
`neb_analysis.py` exactly): `initial`, `intermediate`, `final`.


## 2. Using FPBench Data or Another NEB Dataset

| Workflow | Structural input | Results branch |
|---|---|---|
| Full FP-NEB workflow | Unrelaxed source endpoint structures | `full_fp_neb` |
| Static FP evaluations on the DFT-NEB image structures | Finalized DFT-NEB image structures | `fp_static_on_dft_neb` |

These two input sources are never interchanged. Generated checkpoints are
merged into one standardized all-FP results JSON, loaded directly by the
analysis workflow (`neb_analysis.load_neb_datasets`) and by the companion
DFT-static diagnostic notebook, with no reshaping.

**Required input file.** `DFT_REFERENCE_FILE` (Configuration), following
the schema below. `REFERENCE_MODE` controls validation strictness:
`"fpbench"` requires exactly the FPBench population (154 pathways, 7
finalized DFT-NEB images each); `"external"` derives the active population
from the reference file itself and validates structure, so any correctly
shaped dataset works without weakening the FPBench checks.

```python
reference_data = {
    "common_pathway_keys": [...],
    "pathways": {
        pathway_key: {
            "identifiers": {
                "pathway_key": pathway_key,
                "icsd_id": "...",
                "source_path_id": "...",
            },
            "full_fp_neb_input": {
                "source_start_endpoint_structure": {...},
                "source_end_endpoint_structure": {...},
            },
            "dft_neb_reference": {
                "images": [
                    {
                        "image_index": 0,
                        "endpoint_role": "initial",
                        "structure": {...},
                        "energy_total_eV": ...,
                        "forces_eV_per_angstrom": [...],
                    },
                ],
            },
        },
    },
}

# REFERENCE_MODE = "external" validates exactly this shape, with no
# FPBench-specific population counts:
# REFERENCE_MODE = "external"
# DFT_REFERENCE_FILE = Path("/path/to/my_reference.json")
```

**Validation function:** `validate_reference` (Load and Validate the DFT
Reference, below). **Main workflow:** Generate Jobs, then one FP / selected
FPs / all FPs via `submission_scripts/{full_fp_neb,fp_static_on_dft_neb}/`,
then Merge Standardized Results. **Detailed schema:** `input_data/README.md`.

One set of canonical record-building functions (below) builds every image
and pathway record used by both protocols, merge validation, and this
documentation, so the written schema and the documented schema cannot
drift apart.

In [1]:
import json

ENDPOINT_ROLES = ("initial", "intermediate", "final")

CALCULATION_STATUS_VALUES = ("completed", "partial", "failed", "interrupted", "missing", "not_run")


def endpoint_role_for(image_index, n_images):
    if image_index == 0:
        return "initial"
    if image_index == n_images - 1:
        return "final"
    return "intermediate"


def make_identifiers(source_identifiers):
    # Opaque, reusable identifiers: the complete identifiers dict from the
    # source reference or source FP result is preserved verbatim. Never
    # split pathway_key to reconstruct icsd_id/source_path_id; both are
    # carried through as separate, already-present fields instead.
    return dict(source_identifiers)


def make_full_fp_neb_image_record(image_index, n_images, structure, fp_energy_total_eV,
                                   fp_forces_eV_per_angstrom, calculation_provenance):
    return {
        "image_index": image_index,
        "endpoint_role": endpoint_role_for(image_index, n_images),
        "structure": structure,
        "fp_energy_total_eV": fp_energy_total_eV,
        "fp_forces_eV_per_angstrom": fp_forces_eV_per_angstrom,
        "energy_unit": "eV",
        "force_unit": "eV/angstrom",
        "calculation_provenance": calculation_provenance,
    }


def make_fp_static_image_record(image_index, n_images, source_dft_neb_structure, fp_energy_total_eV,
                                 fp_forces_eV_per_angstrom, calculation_status, calculation_provenance):
    if calculation_status not in CALCULATION_STATUS_VALUES:
        raise ValueError(f"calculation_status must be one of {CALCULATION_STATUS_VALUES}")
    return {
        "image_index": image_index,
        "endpoint_role": endpoint_role_for(image_index, n_images),
        "source_dft_neb_structure": source_dft_neb_structure,
        "fp_energy_total_eV": fp_energy_total_eV,
        "fp_forces_eV_per_angstrom": fp_forces_eV_per_angstrom,
        "energy_unit": "eV",
        "force_unit": "eV/angstrom",
        "calculation_status": calculation_status,
        "calculation_provenance": calculation_provenance,
    }


# Minimal runnable schema example (synthetic 3-image toy pathway; real
# generation uses these same functions in Sections 8-10).
_example_structure = {"lattice": "placeholder", "sites": []}
_example_full_images = [
    make_full_fp_neb_image_record(i, 3, _example_structure, -10.0 + 0.1 * i, [[0.0, 0.0, 0.0]],
                                   "fp_neb_generation_and_run.ipynb, full_fp_neb")
    for i in range(3)
]
print("Example full_fp_neb image record (index 0):")
print(json.dumps(_example_full_images[0], indent=2))


Example full_fp_neb image record (index 0):
{
  "image_index": 0,
  "endpoint_role": "initial",
  "structure": {
    "lattice": "placeholder",
    "sites": []
  },
  "fp_energy_total_eV": -10.0,
  "fp_forces_eV_per_angstrom": [
    [
      0.0,
      0.0,
      0.0
    ]
  ],
  "energy_unit": "eV",
  "force_unit": "eV/angstrom",
  "calculation_provenance": "fp_neb_generation_and_run.ipynb, full_fp_neb"
}


## 3. Configuration

Real Zaratan paths and SLURM settings. `RELAX_MAX_STEPS = 500` is kept
unchanged: it preserves `matcalc.RelaxCalc`'s legacy default, matching the
established workflow (see the code comment above `RELAX_MAX_STEPS` below
for the verification note and its one open item).

In [2]:
from __future__ import annotations
import os, stat, sys, tempfile, time
from pathlib import Path

import numpy as np

REFERENCE_MODE = "fpbench"       # "fpbench" or "external"
DFT_REFERENCE_FILE = Path("../data/ion_migration_neb_reference.json.gz")
# Accepts .json or .json.gz. To validate against another dataset instead:
# REFERENCE_MODE = "external"
# DFT_REFERENCE_FILE = Path("/path/to/my_reference.json")

# Public safety controls -- a routine "Run All" never generates job
# directories, writes submission scripts, submits anything, or promotes
# a merged results file over the canonical data. Submitting jobs is never
# performed by this notebook itself in any case (Section 11 only writes
# shell scripts you run yourself on your own cluster); promotion into a
# results/ directory is likewise always a manual, external step (see the
# README) -- SUBMIT_JOBS and PROMOTE_RESULTS are recorded here for
# documentation/consistency but gate no code path in this notebook.
GENERATE_JOBS           = False   # set True to actually write job directories (Section 10)
WRITE_SUBMISSION_SCRIPTS = False  # set True to write submission_scripts/ (Section 11)
SUBMIT_JOBS              = False  # informational only -- this notebook never submits jobs
PROMOTE_RESULTS          = False  # informational only -- promotion is always a manual, external step

OUTPUT_BASE     = Path("./runs/generated_fp_neb_jobs")
CHECKPOINT_DIR  = Path("./runs/generated_fp_neb_jobs/checkpoints")
MERGED_OUT_DIR  = Path("./runs/generated_fp_neb_jobs/merged")
SUBMISSION_SCRIPTS_DIR = Path("./submission_scripts")
for d in (OUTPUT_BASE, CHECKPOINT_DIR, MERGED_OUT_DIR, SUBMISSION_SCRIPTS_DIR):
    d.mkdir(parents=True, exist_ok=True)

# Full FP-NEB workflow parameters (established workflow, unchanged).
NIMAGES         = 5       # interpolated intermediate images (7 total pathway images)
RELAX_FMAX      = 0.002   # endpoint relaxation convergence threshold (eV/A)
RELAX_MAX_STEPS = 500     # matcalc.RelaxCalc's legacy default (confirmed
                          # against four locally available matcalc source
                          # copies; not directly inspected on Zaratan itself)
NEB_FMAX        = 0.05    # NEB convergence threshold (eV/A), NEB force projection
NEB_MAX_STEPS   = 1000    # maximum climbing-image NEB optimization steps
AUTOSORT_TOLS   = (0.1, 0.3, 0.5, 0.8, 1.0, 1.2, 1.5, 2.0)  # interpolation recovery tolerances

SLURM_FP = dict(
    account   = "YOUR_SLURM_ACCOUNT",    # EDIT: your cluster allocation/account
    partition = "YOUR_SLURM_PARTITION",  # EDIT: your cluster's partition/queue name
    ntasks    = 1,
    cpus      = 4,
    mem       = "4G",
    time      = "144:00:00",
)

print(f"REFERENCE_MODE={REFERENCE_MODE!r}  DFT_REFERENCE_FILE exists: {DFT_REFERENCE_FILE.exists()}")
print(f"RELAX_FMAX={RELAX_FMAX}  NEB_FMAX={NEB_FMAX}  NEB_MAX_STEPS={NEB_MAX_STEPS}  NIMAGES={NIMAGES}")


REFERENCE_MODE='fpbench'  DFT_REFERENCE_FILE exists: True
RELAX_FMAX=0.002  NEB_FMAX=0.05  NEB_MAX_STEPS=1000  NIMAGES=5


## 4. Foundation Potential Registry

Every entry has a `registry_key` (used for job-directory and submission-script naming, never a display name) and an `output_key` (the canonical foundation-potential identifier written into the results JSON's `models` dict). Both are asserted unique, and MACE-MP0 and MACE-MatPES-PBE are asserted distinct on both axes.

In [3]:
POTENTIAL_REGISTRY = {
    "mace": {
        "output_key":     "MACE-MP0_medium",
        "display_name":   "MACE",
        "site_pkgs":     "/path/to/mace_env/lib/python3.10/site-packages/",
        "venv_activate": "/path/to/mace_env/bin/activate",
        "model_path":    "/path/to/checkpoints/mace-mpa-0-medium.model",
        "import_lines":  "from mace.calculators import MACECalculator",
        "calc_lines":    ('calc = MACECalculator(\n'
                           '    model_paths=[MODEL_PATH],\n'
                           '    device="cpu", default_dtype="float64",\n)'),
    },
    "chgnet": {
        "output_key":     "CHGNET",
        "display_name":   "CHGNet",
        "site_pkgs":     "/path/to/chgnet_env/lib/python3.10/site-packages/",
        "venv_activate": "/path/to/chgnet_env/bin/activate",
        "model_path":    None,
        "import_lines":  "from chgnet.model.dynamics import CHGNetCalculator",
        "calc_lines":    "calc = CHGNetCalculator()",
    },
    "m3gnet_mp": {
        "output_key":     "M3GNET_pes",
        "display_name":   "M3GNet",
        "site_pkgs":     "/path/to/m3gnet_env/lib/python3.10/site-packages/",
        "venv_activate": "/path/to/m3gnet_env/bin/activate",
        "model_path":    "/path/to/checkpoints/M3GNet-MP-2021.2.8-PES/",
        "import_lines":  ("from matgl.ext.ase import PESCalculator\n"
                           "from matgl.utils.io import load_model"),
        "calc_lines":    ('model = load_model(MODEL_PATH)\n'
                           'calc = PESCalculator(potential=model)'),
    },
    "uma": {
        "output_key":     "UMA_s1_p1",
        "display_name":   "UMA",
        "site_pkgs":     "/path/to/uma_env/lib/python3.10/site-packages/",
        "venv_activate": "/path/to/uma_env/bin/activate",
        "model_path":    "/path/to/checkpoints/uma-s-1p1.pt",
        "import_lines":  ("from fairchem.core import FAIRChemCalculator\n"
                           "from fairchem.core.units.mlip_unit import load_predict_unit"),
        "calc_lines":    ('predictor = load_predict_unit(MODEL_PATH, device="cpu")\n'
                           'calc = FAIRChemCalculator(predictor, task_name="omat")'),
    },
    "m3gnet_matpes_pbe": {
        "output_key":     "M3GNET_matpes_PBE",
        "display_name":   "M3GNet-MatPES",
        "site_pkgs":     "/path/to/m3gnet_env/lib/python3.10/site-packages/",
        "venv_activate": "/path/to/m3gnet_env/bin/activate",
        "model_path":    "/path/to/checkpoints/M3GNet-MatPES-PBE-v2025.1-PES/",
        "import_lines":  ("from matgl.ext.ase import PESCalculator\n"
                           "from matgl.utils.io import load_model"),
        "calc_lines":    ('model = load_model(MODEL_PATH)\n'
                           'calc = PESCalculator(potential=model)'),
    },
    "tensornet_pbe": {
        "output_key":     "TensorNET_matpes_PBE",
        "display_name":   "TensorNet-MatPES",
        "site_pkgs":     "/path/to/m3gnet_env/lib/python3.10/site-packages/",
        "venv_activate": "/path/to/m3gnet_env/bin/activate",
        "model_path":    "/path/to/checkpoints/TensorNet-MatPES-PBE-v2025.1-PES/",
        "import_lines":  ("from matgl.ext.ase import PESCalculator\n"
                           "from matgl.utils.io import load_model"),
        "calc_lines":    ('model = load_model(MODEL_PATH)\n'
                           'calc = PESCalculator(potential=model)'),
    },
    "mace_matpes_pbe": {
        "output_key":     "MACE_matpes_pbe",
        "display_name":   "MACE-MatPES",
        "site_pkgs":     "/path/to/mace_env/lib/python3.10/site-packages/",
        "venv_activate": "/path/to/mace_env/bin/activate",
        "model_path":    "/path/to/checkpoints/MACE-matpes-pbe-omat-ft.model",
        "import_lines":  "from mace.calculators import MACECalculator",
        "calc_lines":    ('calc = MACECalculator(\n'
                           '    model_paths=[MODEL_PATH],\n'
                           '    device="cpu", default_dtype="float64",\n)'),
    },
}

ACTIVE_FP_KEYS = [
    "mace",
    "chgnet",
    "m3gnet_mp",
    "uma",
    "m3gnet_matpes_pbe",
    "tensornet_pbe",
    "mace_matpes_pbe",
]

assert set(ACTIVE_FP_KEYS) == set(POTENTIAL_REGISTRY.keys())
_output_keys = [POTENTIAL_REGISTRY[k]["output_key"] for k in ACTIVE_FP_KEYS]
assert len(set(_output_keys)) == len(_output_keys), f"Duplicate output_key: {_output_keys}"
assert POTENTIAL_REGISTRY["mace"]["output_key"] != POTENTIAL_REGISTRY["mace_matpes_pbe"]["output_key"]
assert "mace" != "mace_matpes_pbe"

for reg_key in ACTIVE_FP_KEYS:
    pot = POTENTIAL_REGISTRY[reg_key]
    print(f"  {reg_key:20s} -> output_key={pot['output_key']:22s} display_name={pot['display_name']}")


  mace                 -> output_key=MACE-MP0_medium        display_name=MACE
  chgnet               -> output_key=CHGNET                 display_name=CHGNet
  m3gnet_mp            -> output_key=M3GNET_pes             display_name=M3GNet
  uma                  -> output_key=UMA_s1_p1              display_name=UMA
  m3gnet_matpes_pbe    -> output_key=M3GNET_matpes_PBE      display_name=M3GNet-MatPES
  tensornet_pbe        -> output_key=TensorNET_matpes_PBE   display_name=TensorNet-MatPES
  mace_matpes_pbe      -> output_key=MACE_matpes_pbe        display_name=MACE-MatPES


## 5. Load and Validate the DFT Reference

`REFERENCE_MODE == "fpbench"` requires exactly 154 active pathways, 7 finalized DFT-NEB images per pathway, 1078 images total. `REFERENCE_MODE == "external"` derives the active population from the reference file itself: every active key must exist exactly once, each DFT-NEB pathway must have a nonempty, uniquely indexed, consecutively ordered image set, and endpoint roles, structures, energies, forces, atom counts, atom ordering, and units are all validated. Neither mode weakens the other; `"fpbench"` never runs the relaxed external checks in place of its own.

In [4]:
def load_reference(reference_path):
    reference_path = Path(reference_path)
    if reference_path.suffix == ".gz":
        import gzip
        with gzip.open(reference_path, "rt") as f:
            return json.load(f)
    with open(reference_path) as f:
        return json.load(f)


def iter_pathways(reference_data, only_active=True):
    keys = reference_data["common_pathway_keys"] if only_active else list(reference_data["pathways"].keys())
    for pkey in keys:
        yield pkey, reference_data["pathways"][pkey]


def get_full_fp_neb_input(pathway):
    inp = pathway["full_fp_neb_input"]
    return inp["source_start_endpoint_structure"], inp["source_end_endpoint_structure"]


def get_dft_neb_images(pathway):
    return sorted(pathway["dft_neb_reference"]["images"], key=lambda im: im["image_index"])


def _is_valid_structure_dict(struct):
    if not isinstance(struct, dict):
        return False
    sites = struct.get("sites")
    lattice = struct.get("lattice")
    if not isinstance(sites, list) or not sites:
        return False
    if not isinstance(lattice, dict) or "matrix" not in lattice:
        return False
    return True


def _structure_species(struct):
    try:
        return [s["species"][0]["element"] for s in struct["sites"]]
    except (KeyError, IndexError, TypeError):
        return None


def validate_reference(reference_data, mode):
    issues = []

    def issue(msg):
        issues.append(msg)

    active_keys = reference_data.get("common_pathway_keys")
    if not active_keys:
        issue("reference has no common_pathway_keys")
        return issues
    if len(set(active_keys)) != len(active_keys):
        issue("common_pathway_keys contains duplicates")

    missing_from_file = [pk for pk in active_keys if pk not in reference_data.get("pathways", {})]
    if missing_from_file:
        issue(f"{len(missing_from_file)} active keys missing from pathways: {missing_from_file[:5]}")

    image_counts = {}
    for pkey in active_keys:
        pdata = reference_data["pathways"].get(pkey)
        if pdata is None:
            continue
        images = pdata.get("dft_neb_reference", {}).get("images", [])
        image_counts[pkey] = len(images)

        indices = sorted(im["image_index"] for im in images)
        if not images:
            issue(f"{pkey}: dft_neb_reference.images is empty")
        elif indices != list(range(len(images))):
            issue(f"{pkey}: image indices not uniquely 0..N-1 consecutive: {indices}")
        else:
            roles = {im["image_index"]: im.get("endpoint_role") for im in images}
            if roles.get(0) != "initial":
                issue(f"{pkey}: image 0 endpoint_role != 'initial'")
            if roles.get(len(images) - 1) != "final":
                issue(f"{pkey}: last image endpoint_role != 'final'")
            for idx in range(1, len(images) - 1):
                if roles.get(idx) != "intermediate":
                    issue(f"{pkey}: image {idx} endpoint_role != 'intermediate'")

        for im in images:
            struct = im.get("structure", {})
            if not _is_valid_structure_dict(struct):
                issue(f"{pkey} image {im.get('image_index')}: structure is not a valid structure dict (missing sites/lattice)")
                continue
            n_sites = len(struct["sites"])
            forces = im.get("forces_eV_per_angstrom", [])
            if len(forces) != n_sites:
                issue(f"{pkey} image {im['image_index']}: atom-count/force-count mismatch ({n_sites} vs {len(forces)})")
            else:
                for atom_idx, fvec in enumerate(forces):
                    if not (isinstance(fvec, (list, tuple)) and len(fvec) == 3
                            and all(isinstance(c, (int, float)) for c in fvec)):
                        issue(f"{pkey} image {im['image_index']} atom {atom_idx}: force vector is not exactly 3 numeric components")
                        break
                    if not all(np.isfinite(c) for c in fvec):
                        issue(f"{pkey} image {im['image_index']} atom {atom_idx}: force vector contains a non-finite value")
                        break
            energy = im.get("energy_total_eV")
            if not isinstance(energy, (int, float)):
                issue(f"{pkey} image {im['image_index']}: energy_total_eV is not numeric")
            elif not np.isfinite(energy):
                issue(f"{pkey} image {im['image_index']}: energy_total_eV is not finite ({energy})")

        full_input = pdata.get("full_fp_neb_input", {})
        start_struct = full_input.get("source_start_endpoint_structure")
        end_struct = full_input.get("source_end_endpoint_structure")
        if not start_struct:
            issue(f"{pkey}: missing full_fp_neb_input.source_start_endpoint_structure")
        elif not _is_valid_structure_dict(start_struct):
            issue(f"{pkey}: source_start_endpoint_structure is not a valid structure dict")
        if not end_struct:
            issue(f"{pkey}: missing full_fp_neb_input.source_end_endpoint_structure")
        elif not _is_valid_structure_dict(end_struct):
            issue(f"{pkey}: source_end_endpoint_structure is not a valid structure dict")
        if start_struct and end_struct and _is_valid_structure_dict(start_struct) and _is_valid_structure_dict(end_struct):
            if len(start_struct["sites"]) != len(end_struct["sites"]):
                issue(f"{pkey}: source_start/end_endpoint_structure atom-count mismatch "
                      f"({len(start_struct['sites'])} vs {len(end_struct['sites'])})")
            else:
                start_species = _structure_species(start_struct)
                end_species = _structure_species(end_struct)
                if start_species is not None and end_species is not None and sorted(start_species) != sorted(end_species):
                    issue(f"{pkey}: source_start/end_endpoint_structure species are not compatible "
                          f"(same multiset of elements expected)")

    units = reference_data.get("units", {})
    if units.get("energy") != "eV":
        issue(f"units.energy != 'eV' (found {units.get('energy')!r})")
    if units.get("forces") != "eV/angstrom":
        issue(f"units.forces != 'eV/angstrom' (found {units.get('forces')!r})")

    if mode == "fpbench":
        n_active = len(active_keys)
        n_images_total = sum(image_counts.values())
        if n_active != 154:
            issue(f"fpbench mode requires 154 active pathways, found {n_active}")
        wrong_count = [pk for pk, n in image_counts.items() if n != 7]
        if wrong_count:
            issue(f"fpbench mode requires 7 images per pathway, {len(wrong_count)} pathways differ: {wrong_count[:5]}")
        if n_images_total != 1078:
            issue(f"fpbench mode requires 1078 total images, found {n_images_total}")
    elif mode != "external":
        raise ValueError(f"Unknown REFERENCE_MODE: {mode!r}")

    return issues


reference_data = load_reference(DFT_REFERENCE_FILE)
reference_issues = validate_reference(reference_data, REFERENCE_MODE)

n_active = len(reference_data["common_pathway_keys"])
n_images_total = sum(len(reference_data["pathways"][pk].get("dft_neb_reference", {}).get("images", []))
                      for pk in reference_data["common_pathway_keys"]
                      if pk in reference_data["pathways"])
print(f"REFERENCE_MODE={REFERENCE_MODE!r}  active pathways={n_active}  active DFT-reference images={n_images_total}")
print(f"Validation issues: {len(reference_issues)}")
for msg in reference_issues[:20]:
    print(" ", msg)
assert not reference_issues, f"{len(reference_issues)} reference validation issues (see above)"
print("Reference validation passed.")


REFERENCE_MODE='fpbench'  active pathways=154  active DFT-reference images=1078
Validation issues: 0
Reference validation passed.


## 6. Result Schema and Calculation Statuses

`CALCULATION_STATUS_VALUES` (Section 2): `not_run`, `missing`, `interrupted`, `failed`, `partial`, `completed`. Convergence is a separate axis, never conflated with execution status: `endpoint_converged` and `neb_converged` are only meaningful when `calculation_status == "completed"`; a completed NEB optimization can have `neb_converged: false`.

`full_fp_neb` writes exactly one *of* `completed` or an unsuccessful status per (FP, pathway); `fp_static_on_dft_neb` evaluates images independently and can be `partial` (some images succeeded, some did not).

**Unsuccessful `full_fp_neb` records are written to a sibling `unsuccessful_pathways` dict, not into `pathways`.** `neb_analysis.py`'s own documentation (the `NEBAnalysisResults` docstring) states it currently supports exactly three `full_fp_neb` states -- converged, non-converged, and a pathway key entirely absent from `pathways` -- and explicitly defers a fourth failed/missing taxonomy until backed by a real example. Concretely, `build_protocol_coverage_table` and three other sites compute non-convergence as `crec.get("neb_converged", True)`; a `failed`/`not_run` record with `neb_converged: None` would evaluate `not None -> True` there and be silently counted as "ran and did not converge" rather than "never ran", corrupting the coverage table and at least three metric computations. `unsuccessful_pathways` is additive (a new sibling key `neb_analysis.py` never reads) so every failed/missing/interrupted/not_run record stays fully traceable in the standardized file without touching the loader.

`fp_static_on_dft_neb` partial/failed pathways stay directly in `pathways`: that protocol's image-count-mismatch handling in `neb_analysis.py` (`validate_analysis_coverage` / `coverage_mismatch_df`) is already built and tested (`test_neb_analysis.py` cases 14-16, 21) to catch incomplete per-pathway image sets without corrupting other metrics.

In [5]:
def make_neb_status(calculation_status, optimizer_steps=None, log_entry_count=None,
                     neb_converged=None, last_fmax=None,
                     endpoint_converged=None, endpoint_start_fmax=None, endpoint_end_fmax=None,
                     error=None):
    if calculation_status not in CALCULATION_STATUS_VALUES:
        raise ValueError(f"calculation_status must be one of {CALCULATION_STATUS_VALUES}, got {calculation_status!r}")
    if calculation_status != "completed":
        if neb_converged is not None or endpoint_converged is not None:
            raise ValueError("neb_converged/endpoint_converged must be None unless calculation_status == 'completed'")
    else:
        if neb_converged is None or endpoint_converged is None:
            raise ValueError("neb_converged and endpoint_converged must be True/False when calculation_status == 'completed'")
    return {
        "calculation_status": calculation_status,
        "neb_converged": neb_converged,
        "optimizer_steps": optimizer_steps,
        "log_entry_count": log_entry_count,
        "last_fmax_eV_per_angstrom": last_fmax,
        "endpoint_converged": endpoint_converged,
        "endpoint_start_fmax_eV_per_angstrom": endpoint_start_fmax,
        "endpoint_end_fmax_eV_per_angstrom": endpoint_end_fmax,
        "error": error,
    }


def make_unsuccessful_pathway_record(identifiers, calculation_status, error=None):
    if calculation_status not in ("not_run", "missing", "interrupted", "failed"):
        raise ValueError("make_unsuccessful_pathway_record is for not_run/missing/interrupted/failed only")
    return {
        "identifiers": make_identifiers(identifiers),
        "calculation_status": calculation_status,
        "error": error,
    }


print("CALCULATION_STATUS_VALUES:", CALCULATION_STATUS_VALUES)
_demo_completed = make_neb_status("completed", optimizer_steps=12, log_entry_count=13,
                                   neb_converged=False, last_fmax=0.09,
                                   endpoint_converged=True, endpoint_start_fmax=0.001, endpoint_end_fmax=0.0015)
print("Example: completed but non-converged is valid and preserved:", _demo_completed["calculation_status"],
      _demo_completed["neb_converged"])


CALCULATION_STATUS_VALUES: ('completed', 'partial', 'failed', 'interrupted', 'missing', 'not_run')
Example: completed but non-converged is valid and preserved: completed False


## 7. Runtime Checkpointing and Resume Behavior

The **runtime checkpoint** (`runtime_checkpoint.json`, one per job directory) is written by an ASE optimizer observer attached with `optimizer.attach(callback, interval=1)`, updated atomically (temp file plus `os.replace`) on every optimizer step during both endpoint relaxation and the NEB phase, with pathway/FP identifiers, the current step, the actual current fmax (`neb.get_forces()` during the NEB phase, the same quantity the optimizer itself minimizes), current endpoint/NEB status, recoverable current image structures, and a timestamp. It does not itself resume the optimizer's internal state; a resumed run restarts from the last checkpointed structure.

`output.json` (once written) is what determines a completed result; `runtime_checkpoint.json` alone (no `output.json` yet) identifies interrupted work -- both are read directly from each job directory by `determine_full_fp_neb_status`/`determine_fp_static_status` below, which is what Section 10's regeneration skip and the submission scripts' (`submission_scripts/{full_fp_neb,fp_static_on_dft_neb}/`) result-aware resume rules actually use. Merging (Section 12) scans these same per-job outputs directly; there is no separate notebook-level aggregate-completion record read by any of this.

In [6]:
def atomic_write_json(path, data):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    fd, tmp_path = tempfile.mkstemp(dir=str(path.parent), prefix=path.name + ".tmp")
    try:
        with os.fdopen(fd, "w") as f:
            json.dump(data, f, indent=2)
            f.flush()
            os.fsync(f.fileno())
        os.replace(tmp_path, path)
    except Exception:
        if os.path.exists(tmp_path):
            os.remove(tmp_path)
        raise


def load_checkpoint(checkpoint_path):
    checkpoint_path = Path(checkpoint_path)
    if not checkpoint_path.exists():
        return {"records": {}}
    with open(checkpoint_path) as f:
        return json.load(f)


def checkpoint_record_key(output_key, pathway_key):
    return f"{output_key}|{pathway_key}"


def determine_full_fp_neb_status(job_dir):
    # completed/failed come from output.json's own neb_status.calculation_status.
    # interrupted: runtime_checkpoint.json exists but output.json does not.
    # missing: neither exists but the job was generated (run.py present).
    # not_run: the job directory itself does not exist.
    job_dir = Path(job_dir)
    output_path = job_dir / "output.json"
    runtime_ckpt_path = job_dir / "runtime_checkpoint.json"
    if not job_dir.exists():
        return "not_run", None
    if output_path.exists():
        try:
            with open(output_path) as f:
                record = json.load(f)
            return record["neb_status"]["calculation_status"], record
        except Exception as e:
            return "failed", {"error": f"unreadable output.json: {e}"}
    if runtime_ckpt_path.exists():
        with open(runtime_ckpt_path) as f:
            ckpt = json.load(f)
        return "interrupted", ckpt
    if (job_dir / "run.py").exists():
        return "missing", None
    return "not_run", None


def determine_fp_static_status(job_dir):
    job_dir = Path(job_dir)
    output_path = job_dir / "output.json"
    if not job_dir.exists():
        return "not_run", None
    if output_path.exists():
        try:
            with open(output_path) as f:
                record = json.load(f)
            return record.get("calculation_status", "failed"), record
        except Exception as e:
            return "failed", {"error": f"unreadable output.json: {e}"}
    if (job_dir / "run.py").exists():
        return "missing", None
    return "not_run", None


print("Checkpoint helpers ready. Status vocabulary:", CALCULATION_STATUS_VALUES)


Checkpoint helpers ready. Status vocabulary: ('completed', 'partial', 'failed', 'interrupted', 'missing', 'not_run')


## 8. Full FP-NEB Job Generation

Fixed cell and volume, ASE BFGS endpoint relaxation (`fmax < RELAX_FMAX`), five interpolated intermediate images (seven total pathway images), climbing-image FP-NEB (ASE BFGS, `fmax < NEB_FMAX`, up to `NEB_MAX_STEPS` steps). Interpolation retries at increasing `autosort_tol` on failure. Convergence is read from `neb.get_forces()` (the quantity the optimizer itself minimizes and logs), not raw per-image forces. `optimizer.nsteps` is used directly for `optimizer_steps`; `log_entry_count` is an independently observed counter from the same attached callback, never inferred.

In [7]:
FULL_FP_NEB_LINES = [
    "#!/usr/bin/env python3",
    "import sys; sys.path.insert(0, '__SITE_PKGS__')",
    "import json, os, tempfile, time, socket, platform, datetime",
    "import numpy as np",
    "from pymatgen.core import Structure",
    "from pymatgen.io.ase import AseAtomsAdaptor",
    "from ase.optimize import BFGS",
    "from ase.mep import NEB",
    "__IMPORT_LINES__",
    "",
    "MODEL_PATH = __MODEL_PATH__",
    "__CALC_LINES__",
    "",
    'OUTPUT_KEY = "__OUTPUT_KEY__"',
    'PATHWAY_KEY = "__PATHWAY_KEY__"',
    "NIMAGES = __NIMAGES__",
    "RELAX_FMAX = __RELAX_FMAX__",
    "RELAX_MAX_STEPS = __RELAX_MAX_STEPS__",
    "NEB_FMAX = __NEB_FMAX__",
    "NEB_MAX_STEPS = __NEB_MAX_STEPS__",
    "AUTOSORT_TOLS = __AUTOSORT_TOLS__",
    'INPUT_PATH = "__INPUT_PATH__"',
    'OUTPUT_PATH = "__OUTPUT_PATH__"',
    'RUNTIME_CHECKPOINT_PATH = "__RUNTIME_CHECKPOINT_PATH__"',
    "",
    "print('BEGIN ' + OUTPUT_KEY + ' ' + PATHWAY_KEY)",
    "",
    "def _pkg_version(name):",
    "    try:",
    "        import importlib.metadata",
    "        return importlib.metadata.version(name)",
    "    except Exception:",
    "        return None",
    "",
    "def base_calculation_metadata():",
    "    return {",
    "        'registry_key': '__REG_KEY__',",
    "        'output_key': OUTPUT_KEY,",
    "        'display_name': '__DISPLAY_NAME__',",
    "        'model_path': MODEL_PATH,",
    "        'environment_path': '__ENV_PATH__',",
    "        'python_version': platform.python_version(),",
    "        'package_versions': {'ase': _pkg_version('ase'), 'pymatgen': _pkg_version('pymatgen')},",
    "        'source_reference_file': '__SOURCE_REFERENCE_FILE__',",
    "        'source_reference_sha256': '__SOURCE_REFERENCE_SHA256__',",
    "        'protocol': 'full_fp_neb',",
    "        'endpoint_fmax_eV_per_angstrom': RELAX_FMAX,",
    "        'endpoint_max_steps': RELAX_MAX_STEPS,",
    "        'neb_fmax_eV_per_angstrom': NEB_FMAX,",
    "        'neb_max_steps': NEB_MAX_STEPS,",
    "        'n_intermediate_images': NIMAGES,",
    "        'climbing_image': True,",
    "        'optimizer': 'ASE BFGS',",
    "        'fixed_cell': True,",
    "        'autosort_tol_used': None,",
    "        'hostname': socket.gethostname(),",
    "        'slurm_job_id': os.environ.get('SLURM_JOB_ID'),",
    "        'calculation_timestamp_utc': datetime.datetime.utcnow().isoformat() + 'Z',",
    "    }",
    "",
    "def atomic_write_json(path, data):",
    "    tmp = path + '.tmp'",
    "    with open(tmp, 'w') as f:",
    "        json.dump(data, f, indent=2)",
    "        f.flush()",
    "        os.fsync(f.fileno())",
    "    os.replace(tmp, path)",
    "",
    "def struct_dicts(atoms_list):",
    "    return [AseAtomsAdaptor.get_structure(a).as_dict() for a in atoms_list]",
    "",
    "def write_runtime_checkpoint(stage, step, fmax, endpoint_converged, neb_converged, atoms_list):",
    "    payload = {",
    "        'output_key': OUTPUT_KEY,",
    "        'pathway_key': PATHWAY_KEY,",
    "        'stage': stage,",
    "        'step': step,",
    "        'fmax_eV_per_angstrom': fmax,",
    "        'endpoint_converged': endpoint_converged,",
    "        'neb_converged': neb_converged,",
    "        'recoverable_images': struct_dicts(atoms_list) if atoms_list else None,",
    "        'checkpoint_timestamp': time.time(),",
    "    }",
    "    atomic_write_json(RUNTIME_CHECKPOINT_PATH, payload)",
    "",
    "with open(INPUT_PATH) as f:",
    "    rec = json.load(f)",
    "start_pmg = Structure.from_dict(rec['start'])",
    "end_pmg = Structure.from_dict(rec['end'])",
    "",
    "result = {'output_key': OUTPUT_KEY, 'pathway_key': PATHWAY_KEY, 'identifiers': rec['identifiers']}",
    "",
    "def relax_endpoint(pmg_struct, tag):",
    "    atoms = AseAtomsAdaptor.get_atoms(pmg_struct)",
    "    atoms.calc = calc",
    "    opt = BFGS(atoms, logfile=tag + '_relax.log')",
    "    observed = {'n': 0}",
    "    def _cb():",
    "        observed['n'] += 1",
    "        current_fmax = float(np.linalg.norm(atoms.get_forces(), axis=1).max())",
    "        write_runtime_checkpoint('endpoint_' + tag, opt.nsteps, current_fmax, None, None, [atoms])",
    "    opt.attach(_cb, interval=1)",
    "    opt.run(fmax=RELAX_FMAX, steps=RELAX_MAX_STEPS)",
    "    final_fmax = float(np.linalg.norm(atoms.get_forces(), axis=1).max())",
    "    return atoms, opt.nsteps, observed['n'], final_fmax, bool(final_fmax < RELAX_FMAX)",
    "",
    "def interpolate_with_recovery(start_struct, end_struct, nimages):",
    "    last_error = None",
    "    for tol in AUTOSORT_TOLS:",
    "        try:",
    "            structs = start_struct.interpolate(",
    "                end_struct, nimages=nimages + 1, interpolate_lattices=False,",
    "                pbc=False, autosort_tol=tol,",
    "            )",
    "            return structs, tol",
    "        except Exception as e:",
    "            last_error = e",
    "    raise RuntimeError('interpolation failed at all autosort_tol values: ' + str(last_error))",
    "",
    "try:",
    "    start_atoms, start_steps, start_log_n, start_fmax, start_conv = relax_endpoint(start_pmg, 'start')",
    "    end_atoms, end_steps, end_log_n, end_fmax, end_conv = relax_endpoint(end_pmg, 'end')",
    "    endpoint_converged = bool(start_conv and end_conv)",
    "    start_rx = AseAtomsAdaptor.get_structure(start_atoms)",
    "    end_rx = AseAtomsAdaptor.get_structure(end_atoms)",
    "",
    "    interp_structs, used_autosort_tol = interpolate_with_recovery(start_rx, end_rx, NIMAGES)",
    "    images = [AseAtomsAdaptor.get_atoms(s) for s in interp_structs]",
    "    for img in images:",
    "        img.calc = calc",
    "",
    "    neb = NEB(images, climb=True, allow_shared_calculator=True)",
    "    optimizer = BFGS(neb, logfile='neb.log')",
    "    observed_neb = {'log_count': 0}",
    "    def _neb_cb():",
    "        observed_neb['log_count'] += 1",
    "        grad = neb.get_forces().ravel()",
    "        current_fmax = float(np.linalg.norm(grad.reshape(-1, 3), axis=1).max())",
    "        write_runtime_checkpoint('neb', optimizer.nsteps, current_fmax, endpoint_converged, None, neb.images)",
    "    optimizer.attach(_neb_cb, interval=1)",
    "    optimizer.run(fmax=NEB_FMAX, steps=NEB_MAX_STEPS)",
    "",
    "    optimizer_steps = optimizer.nsteps",
    "    log_entry_count = observed_neb['log_count']",
    "    final_gradient = neb.get_forces().ravel()",
    "    final_fmax = float(np.linalg.norm(final_gradient.reshape(-1, 3), axis=1).max())",
    "    neb_converged = bool(final_fmax < NEB_FMAX)",
    "",
    "    final_images = {}",
    "    for idx, img in enumerate(neb.images):",
    "        final_images[str(idx)] = {",
    "            'structure': AseAtomsAdaptor.get_structure(img).as_dict(),",
    "            'fp_energy_total_eV': float(img.get_potential_energy()),",
    "            'fp_forces_eV_per_angstrom': img.get_forces().tolist(),",
    "        }",
    "",
    "    result['final_fp_neb_images'] = final_images",
    "    result['neb_status'] = {",
    "        'calculation_status': 'completed',",
    "        'neb_converged': neb_converged,",
    "        'optimizer_steps': optimizer_steps,",
    "        'log_entry_count': log_entry_count,",
    "        'last_fmax_eV_per_angstrom': final_fmax,",
    "        'endpoint_converged': endpoint_converged,",
    "        'endpoint_start_fmax_eV_per_angstrom': start_fmax,",
    "        'endpoint_end_fmax_eV_per_angstrom': end_fmax,",
    "        'endpoint_start_steps': start_steps,",
    "        'endpoint_end_steps': end_steps,",
    "        'autosort_tol_used': used_autosort_tol,",
    "        'error': None,",
    "    }",
    "    _cm = base_calculation_metadata()",
    "    _cm['autosort_tol_used'] = used_autosort_tol",
    "    result['calculation_metadata'] = _cm",
    "    write_runtime_checkpoint('done', optimizer_steps, final_fmax, endpoint_converged, neb_converged, neb.images)",
    "    print('  OK fmax=%.4f converged=%s endpoint_converged=%s steps=%d' % (final_fmax, neb_converged, endpoint_converged, optimizer_steps))",
    "except Exception as e:",
    "    result['final_fp_neb_images'] = {}",
    "    result['neb_status'] = {",
    "        'calculation_status': 'failed',",
    "        'neb_converged': None,",
    "        'optimizer_steps': None,",
    "        'log_entry_count': None,",
    "        'last_fmax_eV_per_angstrom': None,",
    "        'endpoint_converged': None,",
    "        'endpoint_start_fmax_eV_per_angstrom': None,",
    "        'endpoint_end_fmax_eV_per_angstrom': None,",
    "        'endpoint_start_steps': None,",
    "        'endpoint_end_steps': None,",
    "        'autosort_tol_used': None,",
    "        'error': type(e).__name__ + ': ' + str(e),",
    "    }",
    "    result['calculation_metadata'] = base_calculation_metadata()",
    "    print('  ERR ' + str(e))",
    "",
    "atomic_write_json(OUTPUT_PATH, result)",
    "print('END ' + OUTPUT_KEY + ' ' + PATHWAY_KEY + ' -> ' + OUTPUT_PATH)",
]
FULL_FP_NEB_TEMPLATE = chr(10).join(FULL_FP_NEB_LINES) + chr(10)


def build_full_fp_neb_script(pot, reg_key, pathway_key, input_path, output_path, runtime_checkpoint_path):
    mp = repr(pot['model_path']) if pot['model_path'] else 'None'
    return (
        FULL_FP_NEB_TEMPLATE
        .replace('__SITE_PKGS__', pot['site_pkgs'])
        .replace('__IMPORT_LINES__', pot['import_lines'])
        .replace('__MODEL_PATH__', mp)
        .replace('__CALC_LINES__', pot['calc_lines'])
        .replace('__OUTPUT_KEY__', pot['output_key'])
        .replace('__PATHWAY_KEY__', pathway_key)
        .replace('__NIMAGES__', str(NIMAGES))
        .replace('__RELAX_FMAX__', str(RELAX_FMAX))
        .replace('__RELAX_MAX_STEPS__', str(RELAX_MAX_STEPS))
        .replace('__NEB_FMAX__', str(NEB_FMAX))
        .replace('__NEB_MAX_STEPS__', str(NEB_MAX_STEPS))
        .replace('__AUTOSORT_TOLS__', str(AUTOSORT_TOLS))
        .replace('__INPUT_PATH__', input_path)
        .replace('__OUTPUT_PATH__', output_path)
        .replace('__RUNTIME_CHECKPOINT_PATH__', runtime_checkpoint_path)
        .replace('__REG_KEY__', reg_key)
        .replace('__DISPLAY_NAME__', pot['display_name'])
        .replace('__ENV_PATH__', pot['venv_activate'])
        .replace('__SOURCE_REFERENCE_FILE__', str(DFT_REFERENCE_FILE))
        .replace('__SOURCE_REFERENCE_SHA256__', _dft_reference_sha256)
    )

print('full_fp_neb template ready:', len(FULL_FP_NEB_TEMPLATE), 'chars')


full_fp_neb template ready: 7726 chars


## 9. Static FP Evaluations on DFT-NEB Images

Direct calculator-only single-point evaluation: an ASE `Atoms` object with `calc` attached, `get_potential_energy()`/`get_forces()` called directly, no optimizer or relaxation object involved. Each image is evaluated independently; a pathway with some but not all images successfully evaluated is written with `calculation_status: "partial"`, never silently presented as `"completed"`.

In [8]:
STATIC_LINES = [
    "#!/usr/bin/env python3",
    "import sys; sys.path.insert(0, '__SITE_PKGS__')",
    "import json, os, socket, platform, datetime",
    "import numpy as np",
    "from pymatgen.core import Structure",
    "from pymatgen.io.ase import AseAtomsAdaptor",
    "__IMPORT_LINES__",
    "",
    "MODEL_PATH = __MODEL_PATH__",
    "__CALC_LINES__",
    "",
    'OUTPUT_KEY = "__OUTPUT_KEY__"',
    'PATHWAY_KEY = "__PATHWAY_KEY__"',
    'INPUT_PATH = "__INPUT_PATH__"',
    'OUTPUT_PATH = "__OUTPUT_PATH__"',
    "",
    "print('BEGIN ' + OUTPUT_KEY + ' ' + PATHWAY_KEY)",
    "",
    "def _pkg_version(name):",
    "    try:",
    "        import importlib.metadata",
    "        return importlib.metadata.version(name)",
    "    except Exception:",
    "        return None",
    "",
    "calculation_metadata = {",
    "    'registry_key': '__REG_KEY__',",
    "    'output_key': OUTPUT_KEY,",
    "    'display_name': '__DISPLAY_NAME__',",
    "    'model_path': MODEL_PATH,",
    "    'environment_path': '__ENV_PATH__',",
    "    'python_version': platform.python_version(),",
    "    'package_versions': {'ase': _pkg_version('ase'), 'pymatgen': _pkg_version('pymatgen')},",
    "    'source_reference_file': '__SOURCE_REFERENCE_FILE__',",
    "    'source_reference_sha256': '__SOURCE_REFERENCE_SHA256__',",
    "    'protocol': 'fp_static_on_dft_neb',",
    "    'endpoint_fmax_eV_per_angstrom': None,",
    "    'endpoint_max_steps': None,",
    "    'neb_fmax_eV_per_angstrom': None,",
    "    'neb_max_steps': None,",
    "    'n_intermediate_images': None,",
    "    'climbing_image': None,",
    "    'optimizer': None,",
    "    'fixed_cell': True,",
    "    'autosort_tol_used': None,",
    "    'hostname': socket.gethostname(),",
    "    'slurm_job_id': os.environ.get('SLURM_JOB_ID'),",
    "    'calculation_timestamp_utc': datetime.datetime.utcnow().isoformat() + 'Z',",
    "}",
    "",
    "with open(INPUT_PATH) as f:",
    "    rec = json.load(f)",
    "",
    "image_results = {}",
    "image_errors = {}",
    "for idx_str, img in rec['images'].items():",
    "    try:",
    "        pmg = Structure.from_dict(img['structure'])",
    "        atoms = AseAtomsAdaptor.get_atoms(pmg)",
    "        atoms.calc = calc",
    "        energy = float(atoms.get_potential_energy())",
    "        forces = np.asarray(atoms.get_forces(), dtype=float)",
    "        image_results[idx_str] = {",
    "            'fp_energy_total_eV': energy,",
    "            'fp_forces_eV_per_angstrom': forces.tolist(),",
    "        }",
    "    except Exception as e:",
    "        image_errors[idx_str] = type(e).__name__ + ': ' + str(e)",
    "",
    "n_expected = len(rec['images'])",
    "n_ok = len(image_results)",
    "if n_ok == 0:",
    "    calculation_status = 'failed'",
    "elif n_ok < n_expected:",
    "    calculation_status = 'partial'",
    "else:",
    "    calculation_status = 'completed'",
    "",
    "result = {",
    "    'output_key': OUTPUT_KEY,",
    "    'pathway_key': PATHWAY_KEY,",
    "    'identifiers': rec.get('identifiers', {}),",
    "    'images': image_results,",
    "    'image_errors': image_errors,",
    "    'n_expected_images': n_expected,",
    "    'calculation_status': calculation_status,",
    "    'calculation_metadata': calculation_metadata,",
    "}",
    "",
    "with open(OUTPUT_PATH, 'w') as f:",
    "    json.dump(result, f, indent=2)",
    "print('END ' + OUTPUT_KEY + ' ' + PATHWAY_KEY + ' -> ' + OUTPUT_PATH + ' status=' + calculation_status)",
]
STATIC_TEMPLATE = chr(10).join(STATIC_LINES) + chr(10)


def build_static_script(pot, reg_key, pathway_key, input_path, output_path):
    mp = repr(pot['model_path']) if pot['model_path'] else 'None'
    return (
        STATIC_TEMPLATE
        .replace('__SITE_PKGS__', pot['site_pkgs'])
        .replace('__IMPORT_LINES__', pot['import_lines'])
        .replace('__MODEL_PATH__', mp)
        .replace('__CALC_LINES__', pot['calc_lines'])
        .replace('__OUTPUT_KEY__', pot['output_key'])
        .replace('__PATHWAY_KEY__', pathway_key)
        .replace('__INPUT_PATH__', input_path)
        .replace('__OUTPUT_PATH__', output_path)
        .replace('__REG_KEY__', reg_key)
        .replace('__DISPLAY_NAME__', pot['display_name'])
        .replace('__ENV_PATH__', pot['venv_activate'])
        .replace('__SOURCE_REFERENCE_FILE__', str(DFT_REFERENCE_FILE))
        .replace('__SOURCE_REFERENCE_SHA256__', _dft_reference_sha256)
    )

print('fp_static_on_dft_neb template ready:', len(STATIC_TEMPLATE), 'chars')


fp_static_on_dft_neb template ready: 2882 chars


## 9b. Reproducibility Metadata

`calculation_metadata` is written into every `full_fp_neb` and `fp_static_on_dft_neb` `output.json` (and carried through into the merged file, Section 12), purely additive: registry/environment identity, package versions, the source DFT reference file and its SHA-256, the scientific parameters already fixed above, and the actual execution host/SLURM job ID/timestamp captured by the run script at run time on Zaratan (`socket.gethostname()`, `os.environ.get('SLURM_JOB_ID')`), never a hardcoded string. `neb_analysis.py` never reads this field; older results files without it remain fully loadable.

In [9]:
import hashlib

def _pkg_version(name):
    try:
        import importlib.metadata
        return importlib.metadata.version(name)
    except Exception:
        return None

_dft_reference_sha256 = hashlib.sha256(DFT_REFERENCE_FILE.read_bytes()).hexdigest()
print(f"{DFT_REFERENCE_FILE} sha256={_dft_reference_sha256}")


../data/ion_migration_neb_reference.json.gz sha256=e211dc4a88fe5739218d6b1e47404624225c5a0d787f4dcce3ef61a5574087c4


## 10. Generate Jobs

One script, one SLURM file, and one runtime-checkpoint path per (registry key, pathway), grouped under `chunk_{NN}/`. Job-directory paths use the registry key, never a display name. Regeneration is skipped only for pathways already `calculation_status: "completed"` (full FP-NEB) or `"completed"` (static FP evaluation); this governs whether job *files* are rewritten, not whether a SLURM job is resubmitted (Submission Scripts, below, controls that separately).

In [10]:
N_CHUNKS = 20

def chunk_index_for(position, n_chunks=N_CHUNKS):
    return position % n_chunks


if GENERATE_JOBS:
    full_entries_by_fp = {reg_key: [] for reg_key in ACTIVE_FP_KEYS}
    n_generated, n_skipped_completed = 0, 0

    for position, (pathway_key, pdata) in enumerate(iter_pathways(reference_data)):
        start_struct, end_struct = get_full_fp_neb_input(pdata)
        identifiers = make_identifiers(pdata["identifiers"])
        safe_key = pathway_key.replace("|", "_p")
        chunk_dir_name = f"chunk_{chunk_index_for(position):02d}"

        for reg_key in ACTIVE_FP_KEYS:
            pot = POTENTIAL_REGISTRY[reg_key]
            output_key = pot["output_key"]
            job_dir = OUTPUT_BASE / "full_fp_neb" / reg_key / chunk_dir_name / safe_key
            status, _ = determine_full_fp_neb_status(job_dir)
            if status == "completed":
                n_skipped_completed += 1
                continue

            job_dir.mkdir(parents=True, exist_ok=True)
            input_path = job_dir / "input.json"
            with open(input_path, "w") as f:
                json.dump({"identifiers": identifiers, "start": start_struct, "end": end_struct}, f)

            output_path = job_dir / "output.json"
            runtime_checkpoint_path = job_dir / "runtime_checkpoint.json"
            script_name, slurm_name = "run.py", "submit.slurm"

            py_code = build_full_fp_neb_script(
                pot, reg_key, pathway_key, str(input_path), str(output_path), str(runtime_checkpoint_path)
            )
            (job_dir / script_name).write_text(py_code)
            (job_dir / script_name).chmod((job_dir / script_name).stat().st_mode | stat.S_IXUSR)

            slurm_lines = [
                "#!/bin/bash",
                f"#SBATCH --job-name=fneb_{reg_key}_{safe_key}"[:64],
                f"#SBATCH -A {SLURM_FP['account']}",
                f"#SBATCH --ntasks={SLURM_FP['ntasks']}",
                f"#SBATCH --cpus-per-task={SLURM_FP['cpus']}",
                f"#SBATCH -p {SLURM_FP['partition']}",
                f"#SBATCH --mem-per-cpu={SLURM_FP['mem']}",
                f"#SBATCH -t {SLURM_FP['time']}",
                "#SBATCH --output=slurm_%j.out",
                "",
                f"source {pot['venv_activate']}",
                f"python {script_name}",
            ]
            (job_dir / slurm_name).write_text("\n".join(slurm_lines) + "\n")
            (job_dir / slurm_name).chmod((job_dir / slurm_name).stat().st_mode | stat.S_IXUSR)

            full_entries_by_fp[reg_key].append((str(job_dir), slurm_name))
            n_generated += 1

    print(f"full_fp_neb: generated {n_generated} job dirs across {len(ACTIVE_FP_KEYS)} FPs and {N_CHUNKS} chunks, "
          f"skipped {n_skipped_completed} already-completed")

    static_entries_by_fp = {reg_key: [] for reg_key in ACTIVE_FP_KEYS}
    n_generated_s, n_skipped_s = 0, 0

    for position, (pathway_key, pdata) in enumerate(iter_pathways(reference_data)):
        images = get_dft_neb_images(pdata)
        identifiers = make_identifiers(pdata["identifiers"])
        safe_key = pathway_key.replace("|", "_p")
        chunk_dir_name = f"chunk_{chunk_index_for(position):02d}"

        for reg_key in ACTIVE_FP_KEYS:
            pot = POTENTIAL_REGISTRY[reg_key]
            job_dir = OUTPUT_BASE / "fp_static_on_dft_neb" / reg_key / chunk_dir_name / safe_key
            status, _ = determine_fp_static_status(job_dir)
            if status == "completed":
                n_skipped_s += 1
                continue

            job_dir.mkdir(parents=True, exist_ok=True)
            input_path = job_dir / "input.json"
            with open(input_path, "w") as f:
                json.dump({"identifiers": identifiers,
                           "images": {str(im["image_index"]): {"structure": im["structure"]} for im in images}}, f)

            script_name, slurm_name = "run.py", "submit.slurm"
            py_code = build_static_script(pot, reg_key, pathway_key, str(input_path), str(job_dir / "output.json"))
            (job_dir / script_name).write_text(py_code)
            (job_dir / script_name).chmod((job_dir / script_name).stat().st_mode | stat.S_IXUSR)

            slurm_lines = [
                "#!/bin/bash",
                f"#SBATCH --job-name=fstat_{reg_key}_{safe_key}"[:64],
                f"#SBATCH -A {SLURM_FP['account']}",
                f"#SBATCH --ntasks={SLURM_FP['ntasks']}",
                f"#SBATCH --cpus-per-task={SLURM_FP['cpus']}",
                f"#SBATCH -p {SLURM_FP['partition']}",
                f"#SBATCH --mem-per-cpu={SLURM_FP['mem']}",
                f"#SBATCH -t {SLURM_FP['time']}",
                "#SBATCH --output=slurm_%j.out",
                "",
                f"source {pot['venv_activate']}",
                f"python {script_name}",
            ]
            (job_dir / slurm_name).write_text("\n".join(slurm_lines) + "\n")
            (job_dir / slurm_name).chmod((job_dir / slurm_name).stat().st_mode | stat.S_IXUSR)

            static_entries_by_fp[reg_key].append((str(job_dir), slurm_name))
            n_generated_s += 1

    print(f"fp_static_on_dft_neb: generated {n_generated_s} job dirs, skipped {n_skipped_s} already-completed")
else:
    print("GENERATE_JOBS is False -- skipping job-directory generation. "
          "Set GENERATE_JOBS = True in Configuration to write real job directories "
          "(this can create thousands of files/directories under runs/).")
    full_entries_by_fp = {reg_key: [] for reg_key in ACTIVE_FP_KEYS}
    static_entries_by_fp = {reg_key: [] for reg_key in ACTIVE_FP_KEYS}


GENERATE_JOBS is False -- skipping job-directory generation. Set GENERATE_JOBS = True in Configuration to write real job directories (this can create thousands of files/directories under runs/).


## 11. Submission Scripts

Bash submission interfaces under `submission_scripts/<protocol>/`: one FP, a user-provided list, or all active FPs. FP selection always uses canonical registry keys, validated against a written `fp_keys.txt` manifest; unknown or duplicate keys are rejected before anything is submitted. Every script supports `--dry-run`.

**Full FP-NEB default resume behavior**: skip every `calculation_status: "completed"` record, including `neb_converged: true` and `neb_converged: false`. A completed 1000-step non-converged pathway is a valid, preserved result and is not resubmitted automatically. `--retry-non-converged` resubmits only completed records with `neb_converged: false`. `--force` resubmits everything.

**Static FP evaluation default resume behavior**: skip only fully `completed` pathways; `partial`, `failed`, `interrupted`, and `missing` pathways are resubmitted by default. `--force` resubmits everything including completed pathways.

In [11]:
def write_lines(path, py_lines):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text("\n".join(py_lines) + "\n")
    path.chmod(path.stat().st_mode | stat.S_IXUSR)
    return path


def _key_validation_lines(usage_noun):
    return [
        "usage() {",
        f'  echo "Usage: $0 [--dry-run] [--force] [--retry-non-converged] [--label LABEL] {usage_noun} [{usage_noun} ...]" >&2',
        "  exit 1",
        "}",
        "",
        "while [[ $# -gt 0 ]]; do",
        '  case "$1" in',
        "    --dry-run) DRY_RUN=1; shift ;;",
        "    --force) FORCE=1; shift ;;",
        "    --retry-non-converged) RETRY_NON_CONVERGED=1; shift ;;",
        "    --label) LABEL=\"$2\"; shift 2 ;;",
        "    -h|--help) usage ;;",
        "    *) REQUESTED_KEYS+=(\"$1\"); shift ;;",
        "  esac",
        "done",
        "",
        "if [[ ${#REQUESTED_KEYS[@]} -eq 0 ]]; then usage; fi",
        "",
        "VALID_KEYS=()",
        "while IFS= read -r line; do",
        '  [[ -n "$line" ]] && VALID_KEYS+=("$line")',
        'done < "$FP_KEYS_FILE"',
        "",
        "key_in_list() {",
        '  local needle="$1"; shift',
        "  local candidate",
        '  for candidate in "$@"; do',
        '    [[ "$candidate" == "$needle" ]] && return 0',
        "  done",
        "  return 1",
        "}",
        "",
        'for k in "${REQUESTED_KEYS[@]}"; do',
        '  if ! key_in_list "$k" "${VALID_KEYS[@]}"; then',
        f'    echo "ERROR: unknown {usage_noun} \'$k\'. Valid keys: ${{VALID_KEYS[*]}}" >&2',
        "    exit 1",
        "  fi",
        "  n_occurrences=0",
        '  for k2 in "${REQUESTED_KEYS[@]}"; do',
        '    [[ "$k2" == "$k" ]] && n_occurrences=$((n_occurrences+1))',
        "  done",
        '  if [[ "$n_occurrences" -gt 1 ]]; then',
        f'    echo "ERROR: duplicate {usage_noun} \'$k\' requested" >&2',
        "    exit 1",
        "  fi",
        "done",
        "",
        'if [[ -z "$LABEL" ]]; then LABEL="$(date +%Y%m%d_%H%M%S)"; fi',
        'LOG_DIR="$(cd "$(dirname "${BASH_SOURCE[0]}")" && pwd)/logs/${LABEL}"',
        'mkdir -p "$LOG_DIR"',
        'LOG_FILE="${LOG_DIR}/submission.log"',
    ]


def common_sh_lines_full_fp_neb(job_root, slurm_basename, completion_file):
    lines_ = [
        "#!/bin/bash",
        "set -euo pipefail",
        "",
        'PROTOCOL="full_fp_neb"',
        f'JOB_ROOT="{job_root}"',
        f'SLURM_BASENAME="{slurm_basename}"',
        f'COMPLETION_FILE="{completion_file}"',
        'FP_KEYS_FILE="$(cd "$(dirname "${BASH_SOURCE[0]}")" && pwd)/fp_keys.txt"',
        "",
        "DRY_RUN=0", "FORCE=0", "RETRY_NON_CONVERGED=0", 'LABEL=""', "REQUESTED_KEYS=()",
        "",
    ] + _key_validation_lines("FP_KEY") + [
        "",
        'echo "Run label: $LABEL"',
        'echo "Protocol: $PROTOCOL"',
        'echo "Requested FP keys: ${REQUESTED_KEYS[*]}"',
        'echo "Dry run: $DRY_RUN   Force: $FORCE   Retry non-converged: $RETRY_NON_CONVERGED"',
        "",
        "n_submitted=0",
        "n_skipped=0",
        "",
        'for fp_key in "${REQUESTED_KEYS[@]}"; do',
        '  fp_root="${JOB_ROOT}/${fp_key}"',
        '  if [[ ! -d "$fp_root" ]]; then',
        '    echo "WARNING: no job directory for fp=$fp_key under $fp_root" >&2',
        "    continue",
        "  fi",
        '  while IFS= read -r -d "" slurm_path; do',
        '    job_dir="$(dirname "$slurm_path")"',
        f'    completion_file="${{job_dir}}/{completion_file}"',
        "    is_completed=0",
        "    is_converged=0",
        '    if [[ -f "$completion_file" ]] && grep -q \'"calculation_status": "completed"\' "$completion_file"; then',
        "      is_completed=1",
        '      grep -q \'"neb_converged": true\' "$completion_file" && is_converged=1',
        "    fi",
        "    skip_ok=0",
        '    if [[ "$is_completed" -eq 1 ]]; then',
        '      if [[ "$RETRY_NON_CONVERGED" -eq 1 ]] && [[ "$is_converged" -eq 0 ]]; then',
        "        skip_ok=0   # explicitly retrying this non-converged completed record",
        "      else",
        "        skip_ok=1   # default: skip every validated completed record, converged or not",
        "      fi",
        "    fi",
        '    if [[ "$FORCE" -eq 0 ]] && [[ "$skip_ok" -eq 1 ]]; then',
        "      n_skipped=$((n_skipped+1))",
        "      continue",
        "    fi",
        '    pathway_id="$(basename "$job_dir")"',
        '    if [[ "$DRY_RUN" -eq 1 ]]; then',
        '      echo "[DRY RUN] would submit: protocol=$PROTOCOL fp=$fp_key pathway=$pathway_id dir=$job_dir"',
        "      n_submitted=$((n_submitted+1))",
        "      continue",
        "    fi",
        '    job_id="$(cd "$job_dir" && sbatch "$SLURM_BASENAME" | awk \'{print $NF}\')"',
        '    ts="$(date -u +%Y-%m-%dT%H:%M:%SZ)"',
        '    echo "${ts} protocol=${PROTOCOL} fp=${fp_key} pathway=${pathway_id} dir=${job_dir} slurm_job_id=${job_id}" >> "$LOG_FILE"',
        '    echo "Submitted: fp=$fp_key pathway=$pathway_id slurm_job_id=$job_id"',
        "    n_submitted=$((n_submitted+1))",
        f'  done < <(find "$fp_root" -name "$SLURM_BASENAME" -print0 | sort -z)',
        "done",
        "",
        'echo "Done. Submitted=$n_submitted Skipped(already validated completed)=$n_skipped"',
        'echo "Log: $LOG_FILE"',
    ]
    return lines_


def common_sh_lines_fp_static(job_root, slurm_basename, completion_file):
    lines_ = [
        "#!/bin/bash",
        "set -euo pipefail",
        "",
        'PROTOCOL="fp_static_on_dft_neb"',
        f'JOB_ROOT="{job_root}"',
        f'SLURM_BASENAME="{slurm_basename}"',
        f'COMPLETION_FILE="{completion_file}"',
        'FP_KEYS_FILE="$(cd "$(dirname "${BASH_SOURCE[0]}")" && pwd)/fp_keys.txt"',
        "",
        "DRY_RUN=0", "FORCE=0", "RETRY_NON_CONVERGED=0", 'LABEL=""', "REQUESTED_KEYS=()",
        "",
    ] + _key_validation_lines("FP_KEY") + [
        "",
        'echo "Run label: $LABEL"',
        'echo "Protocol: $PROTOCOL"',
        'echo "Requested FP keys: ${REQUESTED_KEYS[*]}"',
        'echo "Dry run: $DRY_RUN   Force: $FORCE"',
        "",
        "n_submitted=0",
        "n_skipped=0",
        "",
        'for fp_key in "${REQUESTED_KEYS[@]}"; do',
        '  fp_root="${JOB_ROOT}/${fp_key}"',
        '  if [[ ! -d "$fp_root" ]]; then',
        '    echo "WARNING: no job directory for fp=$fp_key under $fp_root" >&2',
        "    continue",
        "  fi",
        '  while IFS= read -r -d "" slurm_path; do',
        '    job_dir="$(dirname "$slurm_path")"',
        f'    completion_file="${{job_dir}}/{completion_file}"',
        "    skip_ok=0",
        '    if [[ -f "$completion_file" ]] && grep -q \'"calculation_status": "completed"\' "$completion_file"; then',
        "      skip_ok=1",
        "    fi",
        '    if [[ "$FORCE" -eq 0 ]] && [[ "$skip_ok" -eq 1 ]]; then',
        "      n_skipped=$((n_skipped+1))",
        "      continue",
        "    fi",
        '    pathway_id="$(basename "$job_dir")"',
        '    if [[ "$DRY_RUN" -eq 1 ]]; then',
        '      echo "[DRY RUN] would submit: protocol=$PROTOCOL fp=$fp_key pathway=$pathway_id dir=$job_dir"',
        "      n_submitted=$((n_submitted+1))",
        "      continue",
        "    fi",
        '    job_id="$(cd "$job_dir" && sbatch "$SLURM_BASENAME" | awk \'{print $NF}\')"',
        '    ts="$(date -u +%Y-%m-%dT%H:%M:%SZ)"',
        '    echo "${ts} protocol=${PROTOCOL} fp=${fp_key} pathway=${pathway_id} dir=${job_dir} slurm_job_id=${job_id}" >> "$LOG_FILE"',
        '    echo "Submitted: fp=$fp_key pathway=$pathway_id slurm_job_id=$job_id"',
        "    n_submitted=$((n_submitted+1))",
        f'  done < <(find "$fp_root" -name "$SLURM_BASENAME" -print0 | sort -z)',
        "done",
        "",
        'echo "Done. Submitted=$n_submitted Skipped(fully completed)=$n_skipped"',
        'echo "Log: $LOG_FILE"',
    ]
    return lines_


def write_protocol_submission_scripts(protocol_dir, common_lines_fn, job_root, slurm_basename,
                                       active_keys, completion_file):
    protocol_dir = Path(protocol_dir)
    protocol_dir.mkdir(parents=True, exist_ok=True)
    (protocol_dir / "fp_keys.txt").write_text("\n".join(active_keys) + "\n")
    write_lines(protocol_dir / "_common.sh", common_lines_fn(job_root, slurm_basename, completion_file))

    write_lines(protocol_dir / "submit_one_fp.sh", [
        "#!/bin/bash", "set -euo pipefail",
        'DIR="$(cd "$(dirname "${BASH_SOURCE[0]}")" && pwd)"',
        "if [[ $# -lt 1 ]]; then",
        '  echo "Usage: $0 FP_KEY [--dry-run] [--force] [--retry-non-converged] [--label LABEL]" >&2',
        "  exit 1", "fi",
        'FP_KEY="$1"; shift',
        'exec "$DIR/_common.sh" "$@" "$FP_KEY"',
    ])
    write_lines(protocol_dir / "submit_selected_fps.sh", [
        "#!/bin/bash", "set -euo pipefail",
        'DIR="$(cd "$(dirname "${BASH_SOURCE[0]}")" && pwd)"',
        "if [[ $# -lt 1 ]]; then",
        '  echo "Usage: $0 FP_KEY [FP_KEY ...] [--dry-run] [--force] [--retry-non-converged] [--label LABEL]" >&2',
        "  exit 1", "fi",
        'exec "$DIR/_common.sh" "$@"',
    ])
    write_lines(protocol_dir / "submit_all_fps.sh", [
        "#!/bin/bash", "set -euo pipefail",
        'DIR="$(cd "$(dirname "${BASH_SOURCE[0]}")" && pwd)"',
        "ALL_KEYS=()",
        "while IFS= read -r line; do",
        '  [[ -n "$line" ]] && ALL_KEYS+=("$line")',
        'done < "$DIR/fp_keys.txt"',
        'exec "$DIR/_common.sh" "$@" "${ALL_KEYS[@]}"',
    ])
    print(f"Wrote submission scripts: {protocol_dir}/{{_common.sh,submit_one_fp.sh,submit_selected_fps.sh,submit_all_fps.sh,fp_keys.txt}}")


if WRITE_SUBMISSION_SCRIPTS:
    write_protocol_submission_scripts(
        SUBMISSION_SCRIPTS_DIR / "full_fp_neb", common_sh_lines_full_fp_neb,
        str((OUTPUT_BASE / "full_fp_neb").resolve()), "submit.slurm", ACTIVE_FP_KEYS, "output.json",
    )
    write_protocol_submission_scripts(
        SUBMISSION_SCRIPTS_DIR / "fp_static_on_dft_neb", common_sh_lines_fp_static,
        str((OUTPUT_BASE / "fp_static_on_dft_neb").resolve()), "submit.slurm", ACTIVE_FP_KEYS, "output.json",
    )

    print()
    print("Example: submission_scripts/full_fp_neb/submit_one_fp.sh mace --dry-run")
    print("Example: submission_scripts/full_fp_neb/submit_selected_fps.sh mace chgnet uma --dry-run")
    print("Example: submission_scripts/full_fp_neb/submit_all_fps.sh --dry-run")
    print("Example: submission_scripts/full_fp_neb/submit_all_fps.sh --retry-non-converged --dry-run")
else:
    print("WRITE_SUBMISSION_SCRIPTS is False -- skipping submission_scripts/ generation. "
          "Set WRITE_SUBMISSION_SCRIPTS = True in Configuration to write them.")


WRITE_SUBMISSION_SCRIPTS is False -- skipping submission_scripts/ generation. Set WRITE_SUBMISSION_SCRIPTS = True in Configuration to write them.


## 12. Merge Standardized Results

Every generated record is passed through the canonical schema-builder functions from Section 2 before being written into the merged file, so the documented schema and the generated records cannot drift apart. `identifiers` is copied verbatim from the reference/checkpoint record; `pathway_key` is treated as an opaque identifier and never split to reconstruct `icsd_id`/`source_path_id` during merge (both already exist as their own fields in `identifiers`).

Merging rejects (raises) a duplicate pathway key about to overwrite an existing record, and an atom-count or atom-order mismatch against the reference structure. Every expected (FP, active pathway) combination gets an explicit record: `completed` (converged or not) and `partial` go into `pathways`; `failed`/`interrupted`/`missing`/`not_run` go into the sibling `unsuccessful_pathways` dict for `full_fp_neb` (Section 6 explains why), or directly into `pathways` for `fp_static_on_dft_neb`.

In [12]:
class MergeError(Exception):
    pass


def validate_structure_atoms(structure_dict, reference_structure_dict, context):
    from pymatgen.core import Structure
    s = Structure.from_dict(structure_dict)
    r = Structure.from_dict(reference_structure_dict)
    if len(s) != len(r):
        raise MergeError(f"{context}: atom count mismatch ({len(s)} vs {len(r)})")
    if [str(sp) for sp in s.species] != [str(sp) for sp in r.species]:
        raise MergeError(f"{context}: atom order/species mismatch")


def new_all_protocol_results_shell(dataset_name):
    return {
        "schema_version": "1.0.0",
        "component": "ion_migration_neb",
        "dataset_name": dataset_name,
        "units": {"energy": "eV", "forces": "eV/angstrom"},
        "models": {
            pot["output_key"]: {
                "display_name": pot["display_name"],
                "full_fp_neb": {"pathways": {}, "unsuccessful_pathways": {}},
                "fp_static_on_dft_neb": {"pathways": {}, "unsuccessful_pathways": {}},
                "dft_static_on_fp_neb": {"pathways": {}},
            }
            for pot in POTENTIAL_REGISTRY.values()
        },
    }


def merge_full_fp_neb(results_data, reg_key, reference_data, report):
    pot = POTENTIAL_REGISTRY[reg_key]
    output_key = pot["output_key"]
    dest = results_data["models"][output_key]["full_fp_neb"]["pathways"]
    unsuccessful = results_data["models"][output_key]["full_fp_neb"]["unsuccessful_pathways"]

    for position, (pathway_key, pdata) in enumerate(iter_pathways(reference_data)):
        safe_key = pathway_key.replace("|", "_p")
        chunk_dir_name = f"chunk_{chunk_index_for(position):02d}"
        job_dir = OUTPUT_BASE / "full_fp_neb" / reg_key / chunk_dir_name / safe_key
        status, record = determine_full_fp_neb_status(job_dir)
        identifiers = pdata["identifiers"]

        if status != "completed":
            if pathway_key in unsuccessful:
                raise MergeError(f"duplicate unsuccessful record for {output_key} {pathway_key}")
            unsuccessful[pathway_key] = make_unsuccessful_pathway_record(
                identifiers, status, error=(record or {}).get("error"))
            continue

        raw_images = record.get("final_fp_neb_images", {})
        n_images = len(raw_images)
        expected_indices = set(str(i) for i in range(NIMAGES + 2))
        if set(raw_images.keys()) != expected_indices:
            report["incomplete_image_sets"].append((output_key, pathway_key, sorted(raw_images.keys())))
            continue

        start_struct, end_struct = get_full_fp_neb_input(pdata)
        validate_structure_atoms(raw_images["0"]["structure"], start_struct, f"{output_key} {pathway_key} image 0")
        validate_structure_atoms(raw_images[str(NIMAGES + 1)]["structure"], end_struct,
                                  f"{output_key} {pathway_key} last image")

        # calculation_provenance is a per-image free-text string, kept
        # environment-neutral here; the real execution host/environment is
        # in calculation_metadata (below), captured at run time by the run
        # script itself (socket.gethostname()/SLURM_JOB_ID), never hardcoded.
        canonical_images = {
            idx_str: make_full_fp_neb_image_record(
                int(idx_str), n_images, img["structure"], img["fp_energy_total_eV"],
                img["fp_forces_eV_per_angstrom"],
                calculation_provenance=f"full_fp_neb run.py, FP={output_key}",
            )
            for idx_str, img in raw_images.items()
        }

        if pathway_key in dest:
            raise MergeError(f"refusing to silently overwrite existing merged record for {output_key} {pathway_key}")
        dest[pathway_key] = {
            "identifiers": make_identifiers(identifiers),
            "final_fp_neb_images": canonical_images,
            "neb_status": record["neb_status"],
            "calculation_metadata": record.get("calculation_metadata"),
        }
    return dest, unsuccessful


def merge_fp_static_on_dft_neb(results_data, reg_key, reference_data, report):
    # Only genuinely "completed" pathways (every image succeeded) are
    # written into "pathways", the branch the analysis loader reads.
    # not_run/missing/interrupted/failed/partial pathways are preserved in
    # the sibling "unsuccessful_pathways" branch instead, the same
    # nonintrusive principle already used for full_fp_neb (Section 6): the
    # analysis loader never reads that key, so nothing about it changes
    # current analysis metrics, while every attempt stays traceable in the
    # standardized results JSON itself, not only in an in-memory report.
    pot = POTENTIAL_REGISTRY[reg_key]
    output_key = pot["output_key"]
    dest = results_data["models"][output_key]["fp_static_on_dft_neb"]["pathways"]
    unsuccessful = results_data["models"][output_key]["fp_static_on_dft_neb"]["unsuccessful_pathways"]

    for position, (pathway_key, pdata) in enumerate(iter_pathways(reference_data)):
        safe_key = pathway_key.replace("|", "_p")
        chunk_dir_name = f"chunk_{chunk_index_for(position):02d}"
        job_dir = OUTPUT_BASE / "fp_static_on_dft_neb" / reg_key / chunk_dir_name / safe_key
        status, record = determine_fp_static_status(job_dir)
        identifiers = pdata["identifiers"]

        if status != "completed":
            if pathway_key in unsuccessful:
                raise MergeError(f"duplicate unsuccessful record for {output_key} {pathway_key}")
            unsuccessful[pathway_key] = {
                "identifiers": make_identifiers(identifiers),
                "calculation_status": status,
                "image_errors": (record or {}).get("image_errors", {}),
                "error": (record or {}).get("error"),
                "calculation_provenance": f"fp_static_on_dft_neb run.py, FP={output_key}",
            }
            continue

        dft_images = get_dft_neb_images(pdata)
        n_images = len(dft_images)
        raw_images = record.get("images", {})

        canonical_images = {
            str(im["image_index"]): make_fp_static_image_record(
                im["image_index"], n_images, im["structure"],
                raw_images[str(im["image_index"])]["fp_energy_total_eV"],
                raw_images[str(im["image_index"])]["fp_forces_eV_per_angstrom"], "completed",
                calculation_provenance=f"fp_static_on_dft_neb run.py, FP={output_key}",
            )
            for im in dft_images
        }

        if pathway_key in dest:
            raise MergeError(f"refusing to silently overwrite existing merged record for {output_key} {pathway_key}")
        dest[pathway_key] = {
            "identifiers": make_identifiers(identifiers),
            "images": canonical_images,
            "calculation_metadata": record.get("calculation_metadata"),
        }
    return dest, unsuccessful


merge_report = {"incomplete_image_sets": []}
all_results = new_all_protocol_results_shell("ion_migration_neb_fp_results")

for reg_key in ACTIVE_FP_KEYS:
    output_key = POTENTIAL_REGISTRY[reg_key]["output_key"]
    dest_full, unsuccessful_full = merge_full_fp_neb(all_results, reg_key, reference_data, merge_report)
    dest_static, unsuccessful_static = merge_fp_static_on_dft_neb(all_results, reg_key, reference_data, merge_report)
    print(f"  [merge] {output_key:22s} full_fp_neb completed={len(dest_full):3d} unsuccessful={len(unsuccessful_full):3d}  "
          f"fp_static_on_dft_neb completed={len(dest_static):3d} unsuccessful={len(unsuccessful_static):3d}")

assert set(all_results["models"].keys()) == {p["output_key"] for p in POTENTIAL_REGISTRY.values()}
print(f"\nMerge report: incomplete_image_sets={len(merge_report['incomplete_image_sets'])}")

merged_out_path = MERGED_OUT_DIR / "ion_migration_neb_fp_results.json"
with open(merged_out_path, "w") as f:
    json.dump(all_results, f, indent=2)
print(f"Wrote {merged_out_path}")


  [merge] MACE-MP0_medium        full_fp_neb completed=  0 unsuccessful=154  fp_static_on_dft_neb completed=  0 unsuccessful=154
  [merge] CHGNET                 full_fp_neb completed=  0 unsuccessful=154  fp_static_on_dft_neb completed=  0 unsuccessful=154
  [merge] M3GNET_pes             full_fp_neb completed=  0 unsuccessful=154  fp_static_on_dft_neb completed=  0 unsuccessful=154
  [merge] UMA_s1_p1              full_fp_neb completed=  0 unsuccessful=154  fp_static_on_dft_neb completed=  0 unsuccessful=154
  [merge] M3GNET_matpes_PBE      full_fp_neb completed=  0 unsuccessful=154  fp_static_on_dft_neb completed=  0 unsuccessful=154
  [merge] TensorNET_matpes_PBE   full_fp_neb completed=  0 unsuccessful=154  fp_static_on_dft_neb completed=  0 unsuccessful=154
  [merge] MACE_matpes_pbe        full_fp_neb completed=  0 unsuccessful=154  fp_static_on_dft_neb completed=  0 unsuccessful=154

Merge report: incomplete_image_sets=0
Wrote runs/generated_fp_neb_jobs/merged/ion_migration_neb_

## 13. Validate Generated Jobs and Results

Generated-file and schema validation, performed before submission; real VASP/SLURM/FP-execution outcomes are recorded only after execution on Zaratan. Validation here uses general checks that hold both before any Zaratan job has run (records are `not_run`/`missing`) and after real results exist (`completed`, `partial`, `failed`, `interrupted`, converged or not may all coexist); nothing here requires zero completed or zero failed records.

In [13]:
checks = []
def check(name, ok, detail=""):
    checks.append({"check": name, "ok": bool(ok), "detail": str(detail)})

check("fpbench reference: 154 active pathways, 1078 active images", n_active == 154 and n_images_total == 1078)
check("registry has 7 potentials with unique output_keys", len(POTENTIAL_REGISTRY) == 7 and len(set(_output_keys)) == 7)
check("MACE-MP0 and MACE-MatPES-PBE have distinct registry and output keys",
      "mace" != "mace_matpes_pbe" and
      POTENTIAL_REGISTRY["mace"]["output_key"] != POTENTIAL_REGISTRY["mace_matpes_pbe"]["output_key"])

full_job_dirs = list((OUTPUT_BASE / "full_fp_neb").glob("*/*/*"))
static_job_dirs = list((OUTPUT_BASE / "fp_static_on_dft_neb").glob("*/*/*"))
check("full_fp_neb job dir count recorded (0 is valid when GENERATE_JOBS=False)", True, len(full_job_dirs))
check("fp_static_on_dft_neb job dir count recorded (0 is valid when GENERATE_JOBS=False)", True, len(static_job_dirs))

if full_job_dirs and static_job_dirs:
    check("job-directory paths use registry keys, not display names",
          all(d.relative_to(OUTPUT_BASE / "full_fp_neb").parts[0] in ACTIVE_FP_KEYS for d in full_job_dirs[:200]))

    def _compile_or_fail(path):
        try:
            compile(path.read_text(), str(path), "exec")
            return None
        except SyntaxError as e:
            return e
    n_bad_py = sum(1 for d in full_job_dirs + static_job_dirs if _compile_or_fail(d / "run.py") is not None)
    check("every generated run.py compiles (full population, not a sample)", n_bad_py == 0, n_bad_py)

    import subprocess
    n_bad_sh = 0
    for d in full_job_dirs + static_job_dirs:
        r = subprocess.run(["bash", "-n", str(d / "submit.slurm")], capture_output=True, text=True)
        if r.returncode != 0:
            n_bad_sh += 1
    check("every generated submit.slurm passes bash -n (full population)", n_bad_sh == 0, n_bad_sh)

    for sh in SUBMISSION_SCRIPTS_DIR.rglob("*.sh"):
        r = subprocess.run(["bash", "-n", str(sh)], capture_output=True, text=True)
        check(f"submission script passes bash -n: {sh.relative_to(SUBMISSION_SCRIPTS_DIR)}", r.returncode == 0, r.stderr[:200])

    check("run.py contains no reference to endpoint_correspondence",
          "endpoint_correspondence" not in (full_job_dirs[0] / "run.py").read_text())
    check("run.py uses neb.get_forces() for NEB fmax", "neb.get_forces()" in (full_job_dirs[0] / "run.py").read_text())
    check("run.py attaches a genuine optimizer observer", "optimizer.attach(" in (full_job_dirs[0] / "run.py").read_text())
    check("run.py restores autosort_tol interpolation recovery", "AUTOSORT_TOLS" in (full_job_dirs[0] / "run.py").read_text())
    check("static run.py performs calculator-only evaluation, no RelaxCalc",
          "RelaxCalc" not in (static_job_dirs[0] / "run.py").read_text()
          and "get_potential_energy" in (static_job_dirs[0] / "run.py").read_text())

else:
    print("No generated job directories found (GENERATE_JOBS was False, or Section 10 has not "
          "been run with GENERATE_JOBS=True yet) -- skipping the generated-file checks below. "
          "This is expected on a fresh clone and is not a failure.")
# Documented schema matches generated records: build one real record via the
# canonical functions and confirm every documented field is present.
_full_schema_fields = {"image_index", "endpoint_role", "structure", "fp_energy_total_eV",
                        "fp_forces_eV_per_angstrom", "energy_unit", "force_unit", "calculation_provenance"}
check("full_fp_neb image record matches documented schema exactly",
      set(_example_full_images[0].keys()) == _full_schema_fields)
_static_example = make_fp_static_image_record(0, 3, _example_structure, -1.0, [[0, 0, 0]], "completed", "test")
_static_schema_fields = {"image_index", "endpoint_role", "source_dft_neb_structure", "fp_energy_total_eV",
                          "fp_forces_eV_per_angstrom", "energy_unit", "force_unit", "calculation_status",
                          "calculation_provenance"}
check("fp_static image record matches documented schema exactly", set(_static_example.keys()) == _static_schema_fields)
check("endpoint_role vocabulary is initial/intermediate/final",
      {endpoint_role_for(0, 5), endpoint_role_for(2, 5), endpoint_role_for(4, 5)} == {"initial", "intermediate", "final"})
check("'partial' is accepted by make_fp_static_image_record",
      make_fp_static_image_record(1, 3, _example_structure, None, None, "partial", "test")["calculation_status"] == "partial")

# Schema/loader compatibility with the real analysis loader, unmodified.
import sys as _sys
_sys.path.insert(0, "../scripts")
import neb_analysis as na

candidate = new_all_protocol_results_shell("candidate_check")
sample_pathway_key, sample_pdata = next(iter(iter_pathways(reference_data)))
sample_images = get_dft_neb_images(sample_pdata)
sample_out_key = POTENTIAL_REGISTRY["mace"]["output_key"]
candidate["models"][sample_out_key]["full_fp_neb"]["pathways"][sample_pathway_key] = {
    "identifiers": make_identifiers(sample_pdata["identifiers"]),
    "final_fp_neb_images": {
        str(im["image_index"]): make_full_fp_neb_image_record(
            im["image_index"], len(sample_images), im["structure"], im["energy_total_eV"],
            im["forces_eV_per_angstrom"], "candidate check")
        for im in sample_images
    },
    "neb_status": {"calculation_status": "completed", "neb_converged": True,
                   "optimizer_steps": 12, "log_entry_count": 13, "last_fmax_eV_per_angstrom": 0.04,
                   "endpoint_converged": True, "endpoint_start_fmax_eV_per_angstrom": 0.001,
                   "endpoint_end_fmax_eV_per_angstrom": 0.001},
}
candidate_path = MERGED_OUT_DIR / "_candidate_check.json"
with open(candidate_path, "w") as f:
    json.dump(candidate, f)
_dft_ref, _fp_res = na.load_neb_datasets(DFT_REFERENCE_FILE, candidate_path)
check("candidate results file loads via neb_analysis.load_neb_datasets", isinstance(_fp_res, dict))
candidate_path.unlink()

# REFERENCE_MODE == "external" works with a synthetic dataset whose pathway
# count differs from 154 (here: 2 pathways, not the FPBench 154). Uses a
# real, valid pymatgen structure dict, not the Section 2 placeholder (which
# has no real lattice/sites and correctly fails structure validation).
from pymatgen.core import Structure as _Structure, Lattice as _ExtLattice
_valid_structure = _Structure(_ExtLattice.cubic(4.0), ["Li", "Cl"], [[0, 0, 0], [0.5, 0.5, 0.5]]).as_dict()
_example_structure = _valid_structure   # supersede the Section 2 schema-only placeholder for structure-validity checks below

_external_ref = {
    "common_pathway_keys": ["EXT1|1", "EXT2|1"],
    "units": {"energy": "eV", "forces": "eV/angstrom"},
    "pathways": {
        "EXT1|1": {
            "identifiers": {"material_id": "EXT1", "pathway_id": "1", "pathway_key": "EXT1|1"},
            "full_fp_neb_input": {"source_start_endpoint_structure": _example_structure,
                                   "source_end_endpoint_structure": _example_structure},
            "dft_neb_reference": {"images": [
                {"image_index": 0, "endpoint_role": "initial", "structure": _example_structure,
                 "energy_total_eV": -1.0, "forces_eV_per_angstrom": [[0.0, 0.0, 0.0], [0.0, 0.0, 0.0]]},
                {"image_index": 1, "endpoint_role": "intermediate", "structure": _example_structure,
                 "energy_total_eV": -0.9, "forces_eV_per_angstrom": [[0.0, 0.0, 0.0], [0.0, 0.0, 0.0]]},
                {"image_index": 2, "endpoint_role": "final", "structure": _example_structure,
                 "energy_total_eV": -1.0, "forces_eV_per_angstrom": [[0.0, 0.0, 0.0], [0.0, 0.0, 0.0]]},
            ]},
        },
        "EXT2|1": {
            "identifiers": {"material_id": "EXT2", "pathway_id": "1", "pathway_key": "EXT2|1"},
            "full_fp_neb_input": {"source_start_endpoint_structure": _example_structure,
                                   "source_end_endpoint_structure": _example_structure},
            "dft_neb_reference": {"images": [
                {"image_index": 0, "endpoint_role": "initial", "structure": _example_structure,
                 "energy_total_eV": -1.0, "forces_eV_per_angstrom": [[0.0, 0.0, 0.0], [0.0, 0.0, 0.0]]},
                {"image_index": 1, "endpoint_role": "final", "structure": _example_structure,
                 "energy_total_eV": -1.0, "forces_eV_per_angstrom": [[0.0, 0.0, 0.0], [0.0, 0.0, 0.0]]},
            ]},
        },
    },
}
_external_issues = validate_reference(_external_ref, "external")
check("external mode accepts a well-formed 2-pathway dataset (no 154 requirement)", len(_external_issues) == 0, _external_issues)
_external_ref_bad = json.loads(json.dumps(_external_ref))
_external_ref_bad["pathways"]["EXT1|1"]["dft_neb_reference"]["images"] = []
_bad_issues = validate_reference(_external_ref_bad, "external")
check("external mode rejects an empty image set", len(_bad_issues) > 0)
_fpbench_issues_on_external = validate_reference(_external_ref, "fpbench")
check("fpbench mode correctly rejects the same 2-pathway dataset (154 required)", len(_fpbench_issues_on_external) > 0)

import pandas as pd
report_df = pd.DataFrame(checks)
print(report_df.to_string(index=False))
n_fail = (~report_df["ok"]).sum()
print(f"\n{len(report_df)} checks run, {n_fail} failed.")
assert n_fail == 0, "One or more checks failed -- see table above."


No generated job directories found (GENERATE_JOBS was False, or Section 10 has not been run with GENERATE_JOBS=True yet) -- skipping the generated-file checks below. This is expected on a fresh clone and is not a failure.


                                                                            check   ok detail
                       fpbench reference: 154 active pathways, 1078 active images True       
                                registry has 7 potentials with unique output_keys True       
              MACE-MP0 and MACE-MatPES-PBE have distinct registry and output keys True       
         full_fp_neb job dir count recorded (0 is valid when GENERATE_JOBS=False) True      0
fp_static_on_dft_neb job dir count recorded (0 is valid when GENERATE_JOBS=False) True      0
                       full_fp_neb image record matches documented schema exactly True       
                         fp_static image record matches documented schema exactly True       
                           endpoint_role vocabulary is initial/intermediate/final True       
                             'partial' is accepted by make_fp_static_image_record True       
                  candidate results file loads via neb_analy